# Raspador do SaudeLegis (Ministério da Saúde)

Este notebook demonstra o uso do raspador para buscar legislação sanitária no portal SaudeLegis do Ministério da Saúde.

**Fonte:** [saudelegis.saude.gov.br](https://saudelegis.saude.gov.br/saudelegis/secure/norma/listPublic.xhtml)

**Tipo de dados:** Portarias, resoluções, normas e legislação sanitária

**Requisitos:** Este raspador usa Playwright para automação de navegador.

```bash
pip install "raspe[browser] @ git+https://github.com/bdcdo/raspe.git"
python -m playwright install chromium
```

## Importação

In [1]:
import raspe

## Uso Básico

O termo de busca vai no parâmetro `pesquisa`, que o raspador digita no campo assunto do portal. O nome antigo `assunto` ainda funciona, com aviso de descontinuação.

O raspador abre um Chromium sem janela, faz a busca no portal e percorre todas as páginas de resultado, até o limite de 50 (10 normas por página). Por isso não aceita `paginas`, e cada chamada leva algumas dezenas de segundos.

In [2]:
# Busca simples por um termo
# debug=False: log só no nível INFO, e os HTMLs baixados são apagados ao fim
scraper = raspe.saudelegis(debug=False)
dados = scraper.raspar(pesquisa="doença rara")

print(f"Total de normas encontradas: {len(dados)}")
dados.head()

2026-11-15 15:59:17,488 - raspe.SAUDELEGIS - INFO - Iniciando raspagem Playwright: {'pesquisa': 'doença rara'}


2026-11-15 15:59:18,053 - raspe.SAUDELEGIS - INFO - Browser Playwright iniciado com stealth


2026-11-15 15:59:21,846 - raspe.SAUDELEGIS - INFO - Verificando proteção Cloudflare...


2026-11-15 15:59:21,879 - raspe.SAUDELEGIS - INFO - Página carregada (sem proteção Cloudflare)


2026-11-15 15:59:25,631 - raspe.SAUDELEGIS - INFO - Busca executada


2026-11-15 15:59:27,716 - raspe.SAUDELEGIS - INFO - Total de páginas: 2


... (2 outputs left out)

Processando documentos: 100%|██████████| 2/2 [00:00<00:00, 33.67it/s]


2026-11-15 15:59:30,972 - raspe.SAUDELEGIS - INFO - Raspagem concluída: 14 registros


Total de normas encontradas: 14


,tipo_norma,numero,data_pub,origem,ementa,link_url,termo_busca
0,PRT,10968,22/05/2026,MS/GM,"Altera a Portaria GM/MS nº 10.265, de 26 de fe...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
1,PRT,11280,19/05/2026,MS/GM,"Altera a Portaria GM/MS nº 3.132, de 19 de fev...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
2,PRT,10972,29/04/2026,MS/GM,"Altera a Portaria GM/MS nº 3.132, de 19 de fev...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
3,PRT,10276,17/03/2026,MS/GM,"Institui, no âmbito do Sistema Único de Saúde,...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
4,PRT,10265,27/02/2026,MS/GM,Institui Grupo de Trabalho acerca das Doenças ...,http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara


## Parâmetros Disponíveis

| Parâmetro | Tipo | Descrição |
|-----------|------|------------|
| `pesquisa` | str | Termo de busca no campo assunto |
| `headless` | bool | Se True, executa em modo headless (sem janela) |
| `debug` | bool | Se True, mantém arquivos baixados |

## Colunas Retornadas

| Coluna | Descrição |
|--------|------------|
| `tipo_norma` | Tipo da norma (PRT, RES, etc.) |
| `numero` | Número da norma |
| `data_pub` | Data de publicação |
| `origem` | Órgão de origem |
| `ementa` | Descrição/ementa da norma |
| `link_url` | Link para o texto completo |
| `termo_busca` | Termo de busca que gerou a linha |

Tipos e exemplos de cada coluna estão em [`docs/colunas.md`](../docs/colunas.md#raspesaudelegis).

## Uso Avançado

In [3]:
# Buscar múltiplos termos
termos = ["doença rara", "doenças raras"]

todos_dados = []
for termo in termos:
    scraper = raspe.saudelegis(debug=False)
    df = scraper.raspar(pesquisa=termo)
    df['termo_busca'] = termo
    todos_dados.append(df)

import pandas as pd
dados_consolidados = pd.concat(todos_dados, ignore_index=True)

# Remover duplicatas
dados_unicos = dados_consolidados.drop_duplicates(
    subset=['tipo_norma', 'numero', 'data_pub', 'origem', 'ementa']
)

print(f"Total de normas únicas: {len(dados_unicos)}")
dados_unicos.head()

2026-11-15 15:59:30,988 - raspe.SAUDELEGIS - INFO - Iniciando raspagem Playwright: {'pesquisa': 'doença rara'}


2026-11-15 15:59:31,735 - raspe.SAUDELEGIS - INFO - Browser Playwright iniciado com stealth


2026-11-15 15:59:35,232 - raspe.SAUDELEGIS - INFO - Verificando proteção Cloudflare...


2026-11-15 15:59:35,283 - raspe.SAUDELEGIS - INFO - Página carregada (sem proteção Cloudflare)


2026-11-15 15:59:38,919 - raspe.SAUDELEGIS - INFO - Busca executada


2026-11-15 15:59:40,992 - raspe.SAUDELEGIS - INFO - Total de páginas: 2


... (15 outputs left out)

Processando documentos: 100%|██████████| 6/6 [00:00<00:00, 28.25it/s]


2026-11-15 16:00:10,889 - raspe.SAUDELEGIS - INFO - Raspagem concluída: 56 registros


Total de normas únicas: 70


,tipo_norma,numero,data_pub,origem,ementa,link_url,termo_busca
0,PRT,10968,22/05/2026,MS/GM,"Altera a Portaria GM/MS nº 10.265, de 26 de fe...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
1,PRT,11280,19/05/2026,MS/GM,"Altera a Portaria GM/MS nº 3.132, de 19 de fev...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
2,PRT,10972,29/04/2026,MS/GM,"Altera a Portaria GM/MS nº 3.132, de 19 de fev...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
3,PRT,10276,17/03/2026,MS/GM,"Institui, no âmbito do Sistema Único de Saúde,...",http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara
4,PRT,10265,27/02/2026,MS/GM,Institui Grupo de Trabalho acerca das Doenças ...,http://bvsms.saude.gov.br/bvs/saudelegis/gm/20...,doença rara


In [4]:
# Analisar distribuição por tipo de norma
if 'tipo_norma' in dados.columns and len(dados) > 0:
    print("Distribuição por tipo:")
    print(dados['tipo_norma'].value_counts())

Distribuição por tipo:
tipo_norma
PRT                                 9
Resolução da Diretoria Colegiada    2
COP                                 2
PRI                                 1
Name: count, dtype: int64


## Exportação dos Dados

In [5]:
# Exportar para Excel
# dados.to_excel("saudelegis_resultados.xlsx", index=False)

# Exportar para CSV
# dados.to_csv("saudelegis_resultados.csv", index=False)

## Notas

- Este raspador usa Playwright para automação de navegador, então é mais lento que os raspadores HTTP.
- O Chromium precisa estar instalado (`python -m playwright install chromium`).
- Para ver o navegador durante a coleta, use `raspe.saudelegis(headless=False)`.